# Lab 09 — Deep Research

## Objective

Plan a briefing, compare retrieved evidence and cite evidence gaps.

## Introduction and concept

plan selects dimensions, then research groups matching records by dimension and records missing evidence.

## Real-world use case

A teacher wants a local briefing on when caching helps and what its costs are.

## Technology stack and dependencies

Python standard library. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Question → Research plan → Local source retrieval → Evidence comparison and report


## Step 1 — Implement the components

plan selects dimensions, then research groups matching records by dimension and records missing evidence.

In [1]:
import re

SOURCES = [
    {"id": "N1", "topic": "cache", "dimension": "latency", "finding": "Caching avoids repeated database reads."},
    {"id": "N2", "topic": "cache", "dimension": "freshness", "finding": "Cached records can remain stale until expiry."},
    {"id": "N3", "topic": "index", "dimension": "latency", "finding": "An index can speed up selective queries."},
]

def plan(question):
    tokens = re.findall(r"[a-z]+", question.lower())
    if "cache" in tokens or "caching" in tokens:
        return {"topic": "cache", "dimensions": ("latency", "freshness")}
    if "index" in tokens or "indexes" in tokens:
        return {"topic": "index", "dimensions": ("latency", "freshness")}
    raise ValueError("Choose caching or indexes in the local corpus")

def research(question, sources):
    outline = plan(question)
    report = {"question": question, "evidence": {}, "missing": [], "sources": []}
    for dimension in outline["dimensions"]:
        matches = [
            passage for passage in sources
            if (passage["topic"], passage["dimension"]) == (outline["topic"], dimension)
        ]
        if not matches:
            report["missing"].append(dimension)
            continue
        report["evidence"][dimension] = [
            f'{passage["finding"]} [{passage["id"]}]' for passage in matches
        ]
        report["sources"].extend(passage["id"] for passage in matches)
    return report

## Step 2 — Run sample inputs and inspect actual output

Compare caching latency and freshness notes. The index query demonstrates a missing dimension.

In [2]:
def main():
    result = research("How does caching affect latency and freshness?", SOURCES)
    print("Local notes comparison:", result)
    assert set(result["evidence"]) == {"latency", "freshness"}
    assert result["sources"] == ["N1", "N2"]
    result = research("What are the trade-offs of an index?", SOURCES)
    print("Coverage gap:", result)
    assert result["missing"] == ["freshness"]
    print("PASS: citations and evidence gaps")

main()

Local notes comparison: {'question': 'How does caching affect latency and freshness?', 'evidence': {'latency': ['Caching avoids repeated database reads. [N1]'], 'freshness': ['Cached records can remain stale until expiry. [N2]']}, 'missing': [], 'sources': ['N1', 'N2']}
Coverage gap: {'question': 'What are the trade-offs of an index?', 'evidence': {'latency': ['An index can speed up selective queries. [N3]']}, 'missing': ['freshness'], 'sources': ['N3']}
PASS: citations and evidence gaps


## Testing and validation

Compare caching latency and freshness notes. The index query demonstrates a missing dimension. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

The fixture corpus is local and pre-labeled. External browsing and broad source verification are outside this exercise.

## How this relates to ExamPilot

Partial. ExamPilot retrieves selected notes and checks grounding in backend/app/document_processing/vector_store.py and backend/app/agents/question_generation_agent/question_generation_agent.py.

## Conclusion

Plan a briefing, compare retrieved evidence and cite evidence gaps. Use the trace and assertions to explain which decisions the workflow makes.